# Extracting Toronto Census-Tract Data from the StatCan ADA Profile Bulk File

The source file (`98-401-X2021007_English_CSV_data.csv`) is the full 2021 Census Profile
(product 98-401-X2021007) in long format, covering **every geography in Canada down to the
ADA level** — 16.5 million rows, 2.6 GB. We only need the rows for Toronto's **census tracts
(CTs)**, and only for the ~184 characteristics already curated in `toronto-ada.csv`.

**Strategy** (to keep this tractable on a laptop):

1. Use `98-401-X2021007_Geo_starting_row.CSV` — StatCan's geography index — to find the exact
   line range in the bulk file that covers the **Toronto CMA's** census tracts, without
   scanning the whole 16.5M-row file.
2. Stream just that line range (~3.2M rows) rather than loading it into a DataFrame, keeping
   only rows whose CT id is one of the **622 Toronto CTs** already established in
   `statcan_2021_ct_profile.csv`, and whose characteristic id is one of the ~184 we care about.
3. Re-code each kept row with the short `CHARACTERISTIC_CODE` labels from `toronto-ada.csv`
   and write the result to `toronto-ct-small.csv`, matching that file's schema.

This keeps peak memory to "one line of the source file" plus the (small) filtered output —
never the full 2.6 GB file.


In [1]:
import csv
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT = Path("../..")

CENSUS_DIR = ROOT / "data/clustering_neighbourhoods/census"
BULK_DIR   = CENSUS_DIR / "98-401-X2021007_eng_CSV"

BULK_DATA_CSV = BULK_DIR / "98-401-X2021007_English_CSV_data.csv"        # 2.6 GB, 16.5M rows
GEO_INDEX_CSV = BULK_DIR / "98-401-X2021007_Geo_starting_row.CSV"        # geography -> starting line

CENSUS_ADA_CSV = CENSUS_DIR / "toronto-ada.csv"                          # source of target characteristic ids/codes
CT_PROFILE_CSV = ROOT / "data/toronto_election_turnout/census/processed/ct/statcan_2021_ct_profile.csv"  # source of target CT geo ids

OUTPUT_CSV = CENSUS_DIR / "toronto-ct-small.csv"

# Bulk file is encoded latin-1 (StatCan CSVs commonly are; the geo index has non-ASCII
# place names like "Trois-Rivières" that confirm this).
BULK_ENCODING = "latin-1"

# Toronto's geography, as it appears in the geo index.
TORONTO_CMA_GEO_CODE = "2021S0503535"
TORONTO_CMA_GEO_NAME = "Toronto"
TORONTO_CT_DGUID_PREFIX = "2021S0507535"  # all CTs belonging to the Toronto CMA (535)

# Output schema mirrors toronto-ada.csv exactly.
OUTPUT_COLUMNS = [
    "GEO_NAME",
    "CHARACTERISTIC_ID",
    "CHARACTERISTIC_NAME",
    "CHARACTERISTIC_CODE",
    "C1_COUNT_TOTAL",
    "C10_RATE_TOTAL",
]


## 1. Target characteristics

`toronto-ada.csv` already curates the set of ~184 census characteristics we care about
(population, age structure, household size, income, etc.), each tagged with a short
`CHARACTERISTIC_CODE`. We reuse that mapping rather than re-deriving it, so the CT-level
output lines up with the ADA-level output column-for-column.


In [3]:
# CHARACTERISTIC_ID -> (CHARACTERISTIC_NAME, CHARACTERISTIC_CODE), read once (small file).
ada_df = pd.read_csv(CENSUS_ADA_CSV, dtype=str)
char_lookup = (
    ada_df.drop_duplicates("CHARACTERISTIC_ID")
    .set_index("CHARACTERISTIC_ID")[["CHARACTERISTIC_NAME", "CHARACTERISTIC_CODE"]]
    .to_dict(orient="index")
)
target_characteristic_ids = set(char_lookup.keys())

print(f"Target characteristics: {len(target_characteristic_ids)}")


Target characteristics: 184


## 2. Target census tracts

The Toronto **CMA** in the bulk file spans ~1,227 census tracts — but that includes Mississauga,
Brampton, Markham, Vaughan, and the rest of the CMA, not just the City of Toronto. The project
already has the authoritative list of the **622 CTs that make up the City of Toronto**
(`statcan_2021_ct_profile.csv`, built from the clipped CT boundary file); we reuse its `geo_id`
column as our CT filter instead of re-deriving a City-of-Toronto boundary here.


In [4]:
ct_profile_df = pd.read_csv(CT_PROFILE_CSV, dtype=str, usecols=["geo_id"])
target_ct_geo_ids = set(ct_profile_df["geo_id"])

print(f"Target census tracts: {len(target_ct_geo_ids)}")


Target census tracts: 622


## 3. Locate Toronto's line range in the bulk file

`98-401-X2021007_Geo_starting_row.CSV` lists every geography in the same order it appears in
the bulk file, along with the 1-indexed line number (header = line 1) where that geography's
block of rows begins. Each geography's block is a fixed size (one row per characteristic).

We find the "Toronto" CMA entry, then walk forward while the DGUID still matches the
`2021S0507535*` (Toronto-CMA census tract) prefix to find every CT belonging to it. The line
range we need to stream is from the first CT's start line to the line just before the *next*
geography (whatever it is) begins — this avoids hardcoding a block size.


In [5]:
with open(GEO_INDEX_CSV, encoding=BULK_ENCODING, newline="") as f:
    geo_index_rows = list(csv.DictReader(f))

toronto_idx = next(
    i for i, row in enumerate(geo_index_rows)
    if row["Geo Code"] == TORONTO_CMA_GEO_CODE and row["Geo Name"] == TORONTO_CMA_GEO_NAME
)

# Walk forward while rows are still Toronto-CMA census tracts.
j = toronto_idx + 1
while j < len(geo_index_rows) and geo_index_rows[j]["Geo Code"].startswith(TORONTO_CT_DGUID_PREFIX):
    j += 1

first_ct_start_line = int(geo_index_rows[toronto_idx + 1]["Line Number"])
next_geo_start_line = int(geo_index_rows[j]["Line Number"])  # first line AFTER Toronto's CTs
last_ct_line = next_geo_start_line - 1

num_cma_cts = j - (toronto_idx + 1)
print(f"Toronto CMA census tracts in bulk file: {num_cma_cts}")
print(f"Streaming lines {first_ct_start_line:,} to {last_ct_line:,} "
      f"({last_ct_line - first_ct_start_line + 1:,} rows)")


Toronto CMA census tracts in bulk file: 1227
Streaming lines 10,818,674 to 14,046,910 (3,228,237 rows)


## 4. Stream the relevant slice and filter

We iterate the bulk file line-by-line with the plain `csv` module (not `pandas.read_csv`, which
would need to materialize every column for every row we skip). Rows before the Toronto CMA
block are skipped without parsing; once inside the block we parse each row only far enough to
check `GEO_NAME` (the CT id) and `CHARACTERISTIC_ID` against our two target sets, keeping the
row only on a double match. We stop as soon as we pass the last line we need.


In [6]:
import itertools

rows_to_read = last_ct_line - first_ct_start_line + 1
kept_rows = []

with open(BULK_DATA_CSV, encoding=BULK_ENCODING, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)  # line 1
    col = {name: i for i, name in enumerate(header)}

    # Skip everything before the first Toronto CT row without parsing it into columns.
    # Data row 1 is file line 2, so skip (first_ct_start_line - 2) rows.
    for _ in itertools.islice(reader, first_ct_start_line - 2):
        pass

    for row in tqdm(itertools.islice(reader, rows_to_read), total=rows_to_read, desc="Scanning Toronto CTs"):
        geo_name = row[col["GEO_NAME"]]
        characteristic_id = row[col["CHARACTERISTIC_ID"]]
        if geo_name in target_ct_geo_ids and characteristic_id in target_characteristic_ids:
            lookup = char_lookup[characteristic_id]
            kept_rows.append({
                "GEO_NAME": geo_name,
                "CHARACTERISTIC_ID": characteristic_id,
                "CHARACTERISTIC_NAME": lookup["CHARACTERISTIC_NAME"],
                "CHARACTERISTIC_CODE": lookup["CHARACTERISTIC_CODE"],
                "C1_COUNT_TOTAL": row[col["C1_COUNT_TOTAL"]],
                "C10_RATE_TOTAL": row[col["C10_RATE_TOTAL"]],
            })

print(f"Kept {len(kept_rows):,} rows")


Scanning Toronto CTs:   0%|          | 0/3228237 [00:00<?, ?it/s]

Kept 114,448 rows


## 5. Assemble and save

The filtered result is small (622 CTs × ~184 characteristics ≈ 114k rows), so it's safe to
build a DataFrame from it now. We order by `GEO_NAME` then `CHARACTERISTIC_ID` for readability
and to match `toronto-ada.csv`'s row order.


In [7]:
out_df = pd.DataFrame(kept_rows, columns=OUTPUT_COLUMNS)
out_df["_char_id_sort"] = out_df["CHARACTERISTIC_ID"].astype(int)
out_df = (
    out_df.sort_values(["GEO_NAME", "_char_id_sort"])
    .drop(columns="_char_id_sort")
    .reset_index(drop=True)
)

out_df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(out_df):,} rows to {OUTPUT_CSV}")
out_df.head()


Wrote 114,448 rows to ../../data/clustering_neighbourhoods/census/toronto-ct-small.csv


,GEO_NAME,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,CHARACTERISTIC_CODE,C1_COUNT_TOTAL,C10_RATE_TOTAL
0,5350001.00,1,"Population, 2021",pop_2021,599,
1,5350001.00,2,"Population, 2016",pop_2016,595,
2,5350001.00,3,"Population percentage change, 2016 to 2021",pop_pct_change,0.7,0.7
3,5350001.00,6,Population density per square kilometre,pop_density,87.8,87.8
4,5350001.00,34,Total - Distribution (%) of the population by ...,age_total_dist,100,100


## 6. Sanity checks

Every target CT should have (at most) one row per target characteristic, and every
characteristic/geography we asked for should actually have been found in the bulk file.


In [8]:
found_geo_ids = set(out_df["GEO_NAME"])
found_char_ids = set(out_df["CHARACTERISTIC_ID"])

missing_geo_ids = target_ct_geo_ids - found_geo_ids
missing_char_ids = target_characteristic_ids - found_char_ids
dupes = out_df.duplicated(subset=["GEO_NAME", "CHARACTERISTIC_ID"]).sum()

print(f"CTs found: {len(found_geo_ids)} / {len(target_ct_geo_ids)} (missing: {len(missing_geo_ids)})")
print(f"Characteristics found: {len(found_char_ids)} / {len(target_characteristic_ids)} (missing: {len(missing_char_ids)})")
print(f"Duplicate (GEO_NAME, CHARACTERISTIC_ID) rows: {dupes}")
print(f"Expected row count: {len(target_ct_geo_ids)} x {len(target_characteristic_ids)} = {len(target_ct_geo_ids) * len(target_characteristic_ids):,}")
print(f"Actual row count:   {len(out_df):,}")

assert not missing_geo_ids, f"Missing CTs: {missing_geo_ids}"
assert not missing_char_ids, f"Missing characteristics: {missing_char_ids}"
assert dupes == 0, "Unexpected duplicate rows"


CTs found: 622 / 622 (missing: 0)
Characteristics found: 184 / 184 (missing: 0)
Duplicate (GEO_NAME, CHARACTERISTIC_ID) rows: 0
Expected row count: 622 x 184 = 114,448
Actual row count:   114,448
